In [0]:
accounts_df = spark.read.table("global_tech_bronze.global_tech_schema.raw_accounts")
employee_df = spark.read.table("global_tech_bronze.global_tech_schema.raw_employee")
departments_df = spark.read.table("global_tech_bronze.global_tech_schema.raw_departments")
company_df = spark.read.table("global_tech_bronze.global_tech_schema.raw_company")
payroll_df = spark.read.table("global_tech_bronze.global_tech_schema.raw_payroll")
general_ledgers_df = spark.read.table("global_tech_bronze.global_tech_schema.raw_gl")

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DoubleType, DateType

employee_df.dtypes


In [0]:
from pyspark.sql.functions import trim
accounts_df= accounts_df.withColumn("account_name", trim("account_name"))

In [0]:
display(accounts_df)
# accounts_df.show()

In [0]:
accounts_df.write.mode("overwrite").saveAsTable("global_tech_sliver.global_tech_silver_schema.tf_accounts")

In [0]:
from pyspark.sql.functions import to_date, trim, initcap,col,split, col,round,datediff,when,current_date,monotonically_increasing_id


text_columns = ["first_name", "last_name"]
for col_name in text_columns:
    employee_df = employee_df.withColumn(col_name, (trim(col_name)))

employee_df = employee_df.withColumn("base_salary", round(col("base_salary"), 2))

employee_df = employee_df.withColumn("hire_date", to_date(col("hire_date"), "dd-MM-yyyy HH:mm")) \
       .withColumn("termination_date", to_date(col("termination_date"), "dd-MM-yyyy HH:mm"))\
       .withColumn("tenure_days",
        datediff(
            when(col("termination_date").isNotNull(), col("termination_date"))
            .otherwise(current_date()),
            col("hire_date"))) \
       .withColumn("employee_sk",monotonically_increasing_id())


In [0]:
display(employee_df)

In [0]:
employee_df.write.mode("overwrite").saveAsTable("global_tech_sliver.global_tech_silver_schema.tf_employee")

In [0]:
from pyspark.sql.functions import to_timestamp, round

payroll_df= payroll_df.withColumn("pay_period_start", to_date("pay_period_start", "dd-MM-yyyy HH:mm")) \
         .withColumn("pay_period_end", to_date("pay_period_end", "dd-MM-yyyy HH:mm")) \
         .withColumn("pay_date", to_date("pay_date", "dd-MM-yyyy HH:mm"))

In [0]:
display(payroll_df)

In [0]:
payroll_df.write.mode("overwrite").saveAsTable("global_tech_sliver.global_tech_silver_schema.tf_payroll")

In [0]:
from pyspark.sql.functions import to_timestamp, round, col

general_ledgers_df=general_ledgers_df .withColumn("entry_date", to_date("entry_date", "dd-MM-yyyy HH:mm")) \
         .withColumn("posting_date", to_date("posting_date", "dd-MM-yyyy HH:mm"))

In [0]:
display(general_ledgers_df)

In [0]:
general_ledgers_df.write.mode("overwrite").saveAsTable("global_tech_sliver.global_tech_silver_schema.tf_gl")

In [0]:
from pyspark.sql.functions import to_date

company_df= company_df.withColumn("company_name", trim("company_name"))
company_df = company_df.withColumn("established_date", to_date("established_date", "dd-MM-yyyy"))


In [0]:
display(company_df)
# company_df.show()

In [0]:
company_df.write.mode("overwrite").saveAsTable("global_tech_sliver.global_tech_silver_schema.tf_company")

In [0]:
display(departments_df)